# Notebook 04b – Học Bán Giám Sát (Semi-supervised)
**Đề 2: Dự đoán Bệnh Tim – Kịch bản thiếu nhãn**

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from src.data.loader import load_config
from src.features.builder import FeatureBuilder
from src.models.semi_supervised import SemiSupervisedTrainer
from src.evaluation.report import Reporter
from src.visualization.plots import Plotter

cfg = load_config('../configs/params.yaml')
plotter = Plotter(cfg)
reporter = Reporter(cfg)
print("Modules OK")

## 1. Chuẩn bị dữ liệu

In [ ]:
df = pd.read_parquet('../data/processed/heart_processed.parquet')
builder = FeatureBuilder(cfg)
X, y = builder.get_X_y(df)
X_arr, y_arr = X.values, y.values
seed = cfg['seed']
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, y_arr, test_size=0.2, stratify=y_arr, random_state=seed
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. Thiết kế thực nghiệm

Giả lập kịch bản chỉ có **p% nhãn** (p = 5, 10, 20, 30%):
- Phần còn lại coi là **unlabeled** (gán nhãn -1)
- So sánh **Supervised-only** (chỉ dùng phần có nhãn) vs **Self-Training** (tận dụng unlabeled)
- **Metric ưu tiên:** PR-AUC (vì bài toán y tế cần precision cao)

## 3. Thực nghiệm: Learning Curve theo % nhãn

In [ ]:
trainer_ss = SemiSupervisedTrainer(cfg)
print("Chạy learning curve (Supervised vs Self-Training)...")
print("(Có thể mất vài phút)\n")
curve_df = trainer_ss.learning_curve_by_label_ratio(X_train, y_train, X_test, y_test)
print("\n=== Learning Curve Results ===")
print(curve_df.to_string())

In [ ]:
plotter.plot_learning_curve_semi(curve_df)
reporter.summarize_semi_supervised(curve_df)

## 4. Thực nghiệm với 20% nhãn – so sánh chi tiết

In [ ]:
print("\n=== Thực nghiệm với 20% nhãn ===")
ratio = 0.20
y_partial, labeled_idx = trainer_ss.simulate_partial_labels(X_train, y_train, ratio)
n_labeled = (y_partial != -1).sum()
n_unlabeled = (y_partial == -1).sum()
print(f"Labeled: {n_labeled}, Unlabeled: {n_unlabeled}")

In [ ]:
# 4a. Supervised-only (ít nhãn)
sup_res = trainer_ss.train_supervised_only(X_train, y_partial, X_test, y_test)
print(f"Supervised-only: F1={sup_res['F1']:.4f}, PR-AUC={sup_res['PR-AUC']:.4f}")

In [ ]:
# 4b. Self-Training (SVM)
semi_res_svm = trainer_ss.train_self_training(X_train, y_partial, X_test, y_test, "SVM")
print(f"Self-Training (SVM): F1={semi_res_svm['F1']:.4f}, PR-AUC={semi_res_svm['PR-AUC']:.4f}")

In [ ]:
# 4c. Self-Training (RF)
semi_res_rf = trainer_ss.train_self_training(X_train, y_partial, X_test, y_test, "RF")
print(f"Self-Training (RF): F1={semi_res_rf['F1']:.4f}, PR-AUC={semi_res_rf['PR-AUC']:.4f}")

In [ ]:
# 4d. Label Spreading
try:
    ls_res = trainer_ss.train_label_spreading(X_train, y_partial, X_test, y_test)
    print(f"Label Spreading: F1={ls_res['F1']:.4f}, PR-AUC={ls_res.get('PR-AUC','N/A')}")
except Exception as e:
    print(f"Label Spreading error: {e}")

## 5. Bảng so sánh Supervised vs Semi-supervised

In [ ]:
results_ss = trainer_ss.get_results_df()
print("\n=== Bảng so sánh ===")
print(results_ss.round(4).to_string())
reporter.save_table(results_ss.round(4), 'semi_supervised_comparison_20pct.csv')

## 6. Phân tích rủi ro pseudo-label

In [ ]:
print("\n=== Phân tích rủi ro pseudo-label (20% nhãn) ===")
risk = trainer_ss.analyze_pseudo_label_risk(X_train, y_partial, y_train)
print(f"Unlabeled samples: {risk['n_unlabeled']}")
print(f"Pseudo-labeled: {risk['n_pseudo_labeled']}")
print(f"Sai: {risk['n_wrong_pseudo']} ({risk['error_rate']:.1%})")
print("\nRủi ro chính trong y tế:")
print("  - Pseudo-label sai = False Negative → bỏ sót bệnh nhân có bệnh")
print("  - Nguy hiểm hơn False Positive → cần ngưỡng tin cậy cao (threshold=0.85)")
print("  - Nhóm khó: bệnh nhân có triệu chứng mờ nhạt, atypical angina")

## 7. Kết luận Bán giám sát

| Kịch bản | PR-AUC |
|----------|--------|
| Supervised 5% nhãn | thấp |
| Self-Training 5% nhãn | cải thiện |
| Supervised 30% nhãn | tốt hơn |
| Self-Training 30% nhãn | tốt nhất |

**Nhận xét:**
- Self-Training cải thiện đáng kể khi % nhãn thấp (5-10%)
- Khi có ≥ 20% nhãn, supervised-only đã khá tốt
- Ngưỡng tin cậy 0.85 giúp giảm pseudo-label sai
- **Trong bối cảnh y tế**, nên dùng threshold cao để tránh bỏ sót bệnh

In [ ]:
print("Notebook 04b hoàn tất.")